# 2. Missingness

Continues from [0_load_and_orient.ipynb](0_load_and_orient.ipynb) — reloads
the same setup so this notebook runs standalone.

**Scope:** this notebook only identifies and documents missingness patterns (structural absence vs. truly unknown). It does not fill or transform any values — that happens in a later preprocessing step.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load csv data for both train and test
df_train = pd.read_csv("../../data/train.csv")
df_test = pd.read_csv("../../data/test.csv")

In [ ]:
# missing counts and percent per column for train data
missing_total = df_train.isnull().sum().sort_values(ascending=False)
missing_total_percent = (df_train.isnull().sum() / df_train.isnull().count()).sort_values(ascending=False) * 100
missing_data = pd.concat([missing_total, missing_total_percent], keys=['missing_total', 'missing_total_percent'], axis=1)
missing_data.head(20)

In [ ]:
# filter total > 0
missing_data = missing_data[missing_data["missing_total"] > 0]
missing_data

In [ ]:
len(missing_data)

In [ ]:
# missing counts and percent per column for test data
missing_total_test = df_test.isnull().sum().sort_values(ascending=False)
missing_total_percent_test = (df_test.isnull().sum() / df_test.isnull().count()).sort_values(ascending=False) * 100
missing_data_test = pd.concat([missing_total_test, missing_total_percent_test], keys=['missing_total_test', 'missing_total_percent_test'], axis=1)
missing_data_test.head(20)

In [ ]:
# filter total > 0 for missing test data
missing_data_test = missing_data_test[missing_data_test["missing_total_test"] > 0]
missing_data_test

In [ ]:
len(missing_data_test)

In [ ]:
# verify GarageType NaN really means "no garage"
df_train[df_train["GarageType"].isnull()]

In [ ]:
df_train[df_train["GarageType"].isnull()][["GarageType", "GarageCars", "GarageArea"]].describe()

since max for GarageCars and GarageArea both are 0's. So, GarageType -> NaN means structural absence not unknown data.

#### Bsmt group

In [ ]:
df_train[df_train["BsmtQual"].isnull()][["BsmtQual", "TotalBsmtSF", "BsmtFinSF1", "BsmtUnfSF"]].describe()

Same pattern as 'GarageType', it's structural absence not unknown value.

In [ ]:
# do the 5 Bsmt columns actually move together, like the garage group does?
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
any_bsmt_null = df_train[bsmt_cols].isnull().any(axis=1)
all_bsmt_null = df_train[bsmt_cols].isnull().all(axis=1)
df_train[any_bsmt_null & ~all_bsmt_null][["Id"] + bsmt_cols + ["TotalBsmtSF", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF"]]

**Correction:** unlike the garage group, the 5 Bsmt columns don't always move together. `Id 333` has a real basement (`TotalBsmtSF` 3206, nonzero `BsmtFinSF2`) but `BsmtFinType2` missing; `Id 949` has a real basement (936 sqft) but `BsmtExposure` missing. These 2 rows are true unrecorded values, same shape as the `MasVnrType` exception documented in the "Remaining structural-absence group" section below — not structural absence. Everything else in the group (the other 35–36 rows per column) remains confirmed structural absence.

#### Remaining structural-absence group

In [ ]:
# verify PoolQC NaN really means "no pool"
df_train[df_train["PoolQC"].isnull()][["PoolArea"]].describe()

In [ ]:
# verify FireplaceQu NaN really means "no fireplace"
df_train[df_train["FireplaceQu"].isnull()][["Fireplaces"]].describe()

In [ ]:
# verify MasVnrType NaN really means "no masonry veneer"
df_train[df_train["MasVnrType"].isnull()][["MasVnrArea"]].describe()

In [ ]:
# how many MasVnrType-NaN rows actually have a nonzero MasVnrArea?
mask = df_train["MasVnrType"].isnull() & (df_train["MasVnrArea"] > 0)
mask.sum(), df_train[mask][["MasVnrType", "MasVnrArea"]]

`PoolQC` and `FireplaceQu` check out (companion column is exactly 0 for every NaN row). `MasVnrType` doesn't — 5 rows have nonzero `MasVnrArea`, so it's excluded from this group and handled separately. `Alley`, `Fence`, `MiscFeature` have no companion numeric column; treated as structural absence per the data dictionary's stated `NA` meaning.

#### Garage-quality group

`GarageQual`, `GarageFinish`, `GarageCond`, `GarageYrBlt` — check whether these NaNs line up with the same 81 rows already confirmed as "no garage" (`GarageType.isnull()`).

In [ ]:
df_train["GarageQual"].value_counts()

In [ ]:
df_train[df_train["GarageQual"].isnull()][["GarageFinish", "GarageCond", "GarageYrBlt"]]

In [ ]:
# confirm these NaNs are in exactly the same rows as GarageType, not just the same count
garage_cols = ["GarageQual", "GarageFinish", "GarageCond", "GarageYrBlt"]
df_train[garage_cols].isnull().eq(df_train["GarageType"].isnull(), axis=0).all()

`GarageQual`, `GarageFinish`, `GarageCond`, and `GarageYrBlt` are `True` for all four — their NaNs fall in exactly the same 81 rows as `GarageType`. Same structural-absence pattern as the rest of the garage group.

#### Electrical

In [ ]:
df_train[df_train["Electrical"].isnull()]    

Looks like plain date-entry gap rather than something structural

In [ ]:
df_train["Electrical"].value_counts()

Decision: Needs a fill later with mode imputation since it is only one row missing the value

### Numerical columns

`LotFrontage` and `MasVnrArea` are left — both continuous, so this isn't a "None" story like the categorical groups. The goal here is understanding the missingness (data-entry gap vs. something explainable) and scoping what an eventual imputation would need, not filling anything.

#### LotFrontage

In [ ]:
df_train["LotFrontage"].describe()

In [ ]:
df_train["LotFrontage"].isnull().sum()

In [ ]:
df_train[df_train["LotFrontage"] == 313]

In [ ]:
df_train["LotFrontage"].value_counts()

In [ ]:
# LotConfig
df_train.groupby("LotConfig")["LotFrontage"].apply(lambda x: x.isnull().mean())

In [ ]:
df_train["LotConfig"].value_counts()

In [ ]:
# Neighborhood
missing_by_neighborhood = df_train.groupby("Neighborhood")["LotFrontage"].apply(lambda x: x.isnull().mean()).sort_values(ascending=False)
missing_by_neighborhood

In [ ]:
df_train["Neighborhood"].value_counts()

In [ ]:
# spread/median of LotFrontage within each neighborhood, non-null values only
df_train.groupby("Neighborhood")["LotFrontage"].describe()[["count", "50%", "std"]]

**Decision:** `LotFrontage` missingness is not structural absence — every house has some frontage, it's just unrecorded for ~17.7% of rows. It's not random either: missing rate climbs sharply for irregular lot configurations (`CulDSac` 52%, `FR2` 30%, `Corner` 24% vs. `Inside` 13%), which makes sense since frontage is ambiguous to define for lots that don't front a single straight street segment. `Neighborhood` is a weaker signal than expected — within-neighborhood spread is close to the global spread for most neighborhoods, so a flat neighborhood-median fill wouldn't help much beyond a few unusually uniform neighborhoods (`Blmngtn`, `BrDale`). The two rows with `LotFrontage == 313` are flagged as suspicious (identical value on very different lots) but not confirmed erroneous.

Net: `LotConfig` (possibly combined with `Neighborhood`) looks like a better basis for imputation than a flat or neighborhood-only median — left as a preprocessing decision, not resolved here.

#### MasVnrArea

In [ ]:
# how many MasVnrArea-null rows are also MasVnrType-null?
(df_train["MasVnrArea"].isnull() & df_train["MasVnrType"].isnull()).sum()

In [ ]:
# look at the rows where both MasVnrArea and MasVnrType are null
df_train[df_train["MasVnrArea"].isnull()][["MasVnrType", "MasVnrArea"]]

**Decision:** all 8 `MasVnrArea`-null rows are also `MasVnrType`-null, giving three distinct masonry-veneer cases: ~859 rows genuinely have no veneer (`MasVnrType` NaN, `MasVnrArea` = 0), 5 rows have a recorded area but no type (the exception found in the "Remaining structural-absence group" section above), and these 8 rows have neither field recorded at all. Whether that last group is really "no veneer" with an incomplete entry, or a fuller data-entry gap, isn't resolvable from this data alone — flagged as its own case rather than folded into either of the other two. Fill strategy for all three is deferred to preprocessing.

### Test set verification

Re-run the same companion-column and alignment checks against `df_test` to confirm the train-based verdicts generalize, rather than assuming they do.

#### Garage group

In [ ]:
df_test[df_test["GarageType"].isnull()][["GarageCars", "GarageArea"]].describe()

In [ ]:
# do GarageQual/Finish/Cond/YrBlt align with GarageType.isnull() in test, like they did in train?
garage_cols = ["GarageQual", "GarageFinish", "GarageCond", "GarageYrBlt"]
align = df_test[garage_cols].isnull().eq(df_test["GarageType"].isnull(), axis=0)
mismatch = df_test[~align.all(axis=1)]
mismatch[["Id", "GarageType"] + garage_cols + ["GarageCars", "GarageArea"]]

2 exception rows in test, both `GarageType == "Detchd"` (a real garage), everything else in the group missing: `Id 2127` (has `GarageCars`/`GarageArea` recorded — just quality/finish/condition/year unrecorded) and `Id 2577` (has *nothing* recorded, not even `GarageCars`/`GarageArea` — the only row in either split where even the numeric companions are missing). Both are true missing data despite a confirmed real garage, not structural absence.

#### Bsmt group

In [ ]:
bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
any_bsmt_null = df_test[bsmt_cols].isnull().any(axis=1)
all_bsmt_null = df_test[bsmt_cols].isnull().all(axis=1)
exception_rows = df_test[any_bsmt_null & ~all_bsmt_null]
len(exception_rows), exception_rows[["Id"] + bsmt_cols + ["TotalBsmtSF", "BsmtFinSF1", "BsmtUnfSF"]]

7 exception rows in test (vs. 2 in train) — all have a real basement (nonzero `TotalBsmtSF`, 173–1595 sqft) but at least one of `BsmtQual`/`BsmtCond`/`BsmtExposure` unrecorded. Same failure mode as the train correction above, just more of them. True missing data.

#### PoolQC

In [ ]:
df_test[(df_test["PoolQC"].isnull()) & (df_test["PoolArea"] > 0)][["Id", "PoolArea", "PoolQC"]]

3 exception rows in test — all have a real pool (`PoolArea` 368–561) but `PoolQC` unrecorded. In train, `PoolQC` was clean (0 exceptions among 1453). True missing data in test only.

#### FireplaceQu

In [ ]:
df_test[df_test["FireplaceQu"].isnull()][["Fireplaces"]].describe()

Clean — `Fireplaces` is 0 for all 730 `FireplaceQu`-null rows, same as train. No exceptions.

#### MasVnrType / MasVnrArea

In [ ]:
# same two checks as train: type-null-but-area-nonzero, and both-null
mask_test = df_test["MasVnrType"].isnull() & (df_test["MasVnrArea"] > 0)
both_null_test = df_test["MasVnrType"].isnull() & df_test["MasVnrArea"].isnull()
mask_test.sum(), both_null_test.sum(), df_test[mask_test][["Id", "MasVnrType", "MasVnrArea"]]

Same 3-case shape as train: 3 rows have a recorded area but no type (train had 5), and all 15 `MasVnrArea`-null rows are also `MasVnrType`-null (train had 8) — no new exception type, just different counts.

**Decision — test set verification:** the train-based verdicts mostly hold, but not perfectly, and the gap matters:
- **`FireplaceQu`:** clean on both splits, no exceptions.
- **`MasVnrType`/`MasVnrArea`:** same 3-case shape on both splits, just different counts.
- **Garage group:** clean-aligned in train, but test has 2 exceptions — a real garage (`GarageType` present) with everything else missing, including one row (`Id 2577`) missing even the numeric companions `GarageCars`/`GarageArea`.
- **Bsmt group:** not actually clean in *either* split — 2 exception rows in train (missed until this check), 7 in test — all real basements with partial quality fields unrecorded.
- **`PoolQC`:** clean in train (0 exceptions among 1453), but 3 test rows have a real pool with `PoolQC` unrecorded.

Net: the "all-or-nothing" structural-absence assumption is right for the *large majority* of rows in every group, but every group except `FireplaceQu` has a small number of real exceptions that a blanket `fillna("None")` would misclassify. These exception rows are documented here as a checklist for whoever writes the preprocessing step — they need row-specific handling (e.g. impute from other signals), not the group's default fill.

#### LotFrontage

In [ ]:
# does the LotConfig-driven missingness pattern hold in test too?
df_test.groupby("LotConfig")["LotFrontage"].apply(lambda x: x.isnull().mean()).sort_values(ascending=False)

In [ ]:
df_test["LotConfig"].value_counts()

**Decision:** the `LotConfig`-driven pattern generalizes to test — `CulDSac` is again highest (46%, vs. train's 52%) and `Inside` again lowest (12.7%, matching train almost exactly). Bonus: `FR3` has a bigger sample in test (n=10 vs. train's n=4) and shows 40% missing here, confirming train's `FR3` 0% was small-sample noise rather than a real effect. No `313`-style duplicate-max outlier in test (`LotFrontage` tops out at 200). The `LotFrontage` verdict from the earlier decision above stands confirmed on both splits.

### Test-only columns

14 columns are missing in `df_test` but never appear in `df_train`'s missing list: `MSZoning`, `Utilities`, `Functional`, `SaleType`, `KitchenQual`, `Exterior1st`, `Exterior2nd`, `BsmtFullBath`, `BsmtHalfBath`, `BsmtFinSF1`, `BsmtFinSF2`, `BsmtUnfSF`, `TotalBsmtSF`, `GarageCars`, `GarageArea` (33 total in the test missingness count near the top of this notebook, minus the 19 shared with train). Grouping by what they're likely to be.

#### Bsmt numeric group

In [ ]:
bsmt_num = ["BsmtFullBath", "BsmtHalfBath", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF"]
null_rows = df_test[df_test[bsmt_num].isnull().any(axis=1)]
null_rows[["Id"] + bsmt_num + ["BsmtQual"]]

Both rows (`Id 2121`, `Id 2189`) fall in the already-confirmed "no basement" group (all 5 categorical Bsmt fields also null there). Structural absence — just a data-entry quirk where the numeric fields were left `NaN` instead of `0` like the rest of that group. No new pattern, no exception.

#### Garage numeric group

In [ ]:
df_test[df_test[["GarageCars", "GarageArea"]].isnull().any(axis=1)][["Id", "GarageCars", "GarageArea", "GarageType"]]

Just `Id 2577` — already found in the "Garage group" test-verification section above as the one row missing *everything*, including these numeric companions, despite a real `GarageType`. No new information, same exception.

#### Remaining single-row categorical fields

In [ ]:
for col in ["MSZoning", "Utilities", "Functional", "SaleType", "KitchenQual", "Exterior1st", "Exterior2nd"]:
    print(col, "— missing Ids:", df_test[df_test[col].isnull()]["Id"].tolist(), "| mode:", df_test[col].mode()[0])

**Decision — test-only columns:** none of these are structural absence — every house has a zoning, an exterior, a sale type, etc. All are plain unrecorded values, same shape as `Electrical` in train, each with a strongly dominant category (`MSZoning`→`RL` ~74%, `Utilities`→`AllPub` ~99.9%, `Functional`→`Typ` ~93%, `SaleType`→`WD` ~86%, `KitchenQual`→`TA` ~51%), so mode imputation is a reasonable candidate later. A couple share rows: `Id 1916` is missing both `MSZoning` and `Utilities`; `Id 2217` is missing both `MSZoning` and `Functional`; `Id 2152` is missing both `Exterior1st` and `Exterior2nd` (same house, two related fields) — not a new pattern, just the same row hitting more than one field.

This closes out the test-only columns: `BsmtFullBath`/`BsmtHalfBath`/`Bsmt*SF`/`TotalBsmtSF` and `GarageCars`/`GarageArea` are the same structural-absence rows already found elsewhere in this notebook, and the 7 remaining categorical fields are plain data-entry gaps deferred to preprocessing like `Electrical`.

## Summary

| Column(s) | Train missing | Test missing | Verdict | Notes / exceptions |
|---|---|---|---|---|
| `PoolQC` | 1453 (99.5%) | 1456 (99.8%) | Structural absence | 3 test rows: real pool (`PoolArea` > 0), `PoolQC` unrecorded |
| `MiscFeature` | 1406 (96.3%) | 1408 (96.5%) | Structural absence | No numeric companion; per data dictionary's `NA` meaning |
| `Alley` | 1369 (93.8%) | 1352 (92.7%) | Structural absence | No numeric companion |
| `Fence` | 1179 (80.8%) | 1169 (80.1%) | Structural absence | No numeric companion |
| `MasVnrType` + `MasVnrArea` | 872 / 8 (59.7% / 0.5%) | 894 / 15 (61.3% / 1.0%) | 3-case split | No veneer (~859 train) / recorded area, no type (5 train, 3 test) / neither recorded (8 train, 15 test) |
| `FireplaceQu` | 690 (47.3%) | 730 (50.0%) | Structural absence | Clean on both splits, no exceptions |
| `LotFrontage` | 259 (17.7%) | 227 (15.6%) | Real gap, not structural | Missing rate tracks `LotConfig` (`CulDSac` highest); confirmed on both splits; two `313`-value rows flagged, not confirmed erroneous |
| Garage group (`Type`/`Qual`/`Finish`/`Cond`/`YrBlt`) | 81 each (5.5%) | 76–78 each (5.2–5.3%) | Structural absence | 2 test exceptions: real garage (`GarageType` set) with quality fields unrecorded — one (`Id 2577`) missing even `GarageCars`/`GarageArea` |
| Bsmt group (`Qual`/`Cond`/`Exposure`/`FinType1`/`FinType2`) | 37–38 each (2.5–2.6%) | 42–45 each (2.9–3.1%) | Structural absence | 2 train + 7 test exceptions: real basement (`TotalBsmtSF` > 0) with a quality field unrecorded |
| `Electrical` | 1 (0.07%) | 0 | Real gap | Single row; `SBrkr` dominant (~91%), mode-imputable |
| Bsmt numeric (`FullBath`/`HalfBath`/`Fin1`/`Fin2`/`Unf`/`TotalSF`) | 0 | 1–2 each | Structural absence | Same "no basement" rows as the Bsmt group; numeric fields left `NaN` instead of `0` |
| Garage numeric (`Cars`/`Area`) | 0 | 1 each | Same exception as Garage group | Both from `Id 2577` |
| `MSZoning`/`Utilities`/`Functional`/`SaleType`/`KitchenQual`/`Exterior1st`/`Exterior2nd` | 0 | 1–4 each | Real gap | Plain unrecorded values, each with a strongly dominant category; `Id 2152` hits both `Exterior1st`/`2nd` |

**Bottom line:** the large-percentage columns (`PoolQC` through `FireplaceQu`) are overwhelmingly structural absence and belong in a `fillna("None")`-style step. `LotFrontage`, `Electrical`, and the small test-only categorical fields are genuine unrecorded values needing a real imputation strategy (median/mode/group-based). Every structural-absence group has a handful of real exceptions (rows with a confirmed feature but an unrecorded sub-field) that need row-specific handling rather than the group's default fill — full list is in the sections above. None of this has been filled yet; all decisions here are for the preprocessing step.